# prepare_pods.ipynb

**Run this ONCE, on the first VM.** It installs git, clones/pulls the repo, and
rebuilds the per-VM bundles `Pod_1 … Pod_5` at the `/workspace` root from the single
`Pod/` template (setting each one's `VM_NAME`). Because `/workspace` is shared across
VMs, once this finishes **every VM already has its folder** — VM _N_ just opens
`/workspace/Pod_N`.

So you only ever maintain **one** `Pod/` template + this generator, not 5 copies.
Re-run it any time the template changes to regenerate all bundles.

In [ ]:
# ---- config + git install + clone/pull (setup logic) ----
import os, subprocess

URL = 'https://github.com/Nice9Tian/stable-query-latent.git'
REPO = '/workspace/stable-query-latent'
DEST_ROOT = '/workspace'      # shared FS -> all VMs see the generated Pod_N folders
N_PODS = 5
VM_PREFIX = 'VM'              # Pod_1 -> VM1, ...

def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)

sh('type -p git >/dev/null 2>&1 || (apt-get update && apt-get install -y git)')
if not os.path.isdir(os.path.join(REPO, '.git')):
    sh(f'git clone {URL} {REPO}')
sh(f'cd {REPO} && git remote set-url origin {URL} && git pull origin main')
sh(f'cd {REPO} && git rev-parse --short HEAD')

In [ ]:
# ---- rebuild Pod_1 .. Pod_N at /workspace from the Pod/ template ----
import json, re, shutil
from pathlib import Path

TEMPLATE = Path(REPO) / 'Pod'
NBS_WITH_VMNAME = ['training.ipynb', 'prepare_training.ipynb', 'realtime_reader.ipynb']
_VM = re.compile(r'VM_NAME = "[^"]*"')      # matches whatever the template placeholder is

def set_vm_name(nb_path, vm):
    doc = json.loads(Path(nb_path).read_text(encoding='utf-8'))
    hits = 0
    for cell in doc.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        src = cell['source']
        joined = ''.join(src) if isinstance(src, list) else src
        if _VM.search(joined):
            cell['source'] = _VM.sub(f'VM_NAME = "{vm}"', joined, count=1)
            hits += 1
    Path(nb_path).write_text(json.dumps(doc, ensure_ascii=False, indent=1), encoding='utf-8')
    return hits

assert TEMPLATE.is_dir(), f'template not found: {TEMPLATE} (did the pull succeed?)'
built = []
for n in range(1, N_PODS + 1):
    vm = f'{VM_PREFIX}{n}'
    dest = Path(DEST_ROOT) / f'Pod_{n}'
    shutil.rmtree(dest, ignore_errors=True)          # idempotent: regenerate cleanly
    shutil.copytree(TEMPLATE, dest)
    for nb in NBS_WITH_VMNAME:
        h = set_vm_name(dest / nb, vm)
        assert h == 1, f'{dest / nb}: expected 1 VM_NAME line, found {h}'
    built.append((str(dest), vm))
    print(f'built {dest}  (VM_NAME={vm})', flush=True)

print('\nDone. On each VM open its folder, then run:')
print('   setup.ipynb  ->  prepare_training.ipynb  ->  training.ipynb  ->  check_paralle.ipynb')
for d, vm in built:
    print(f'   {vm}: {d}')